In [1]:
import re
import html
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
INPUT_FILE = "ultimate_combined_speeches.csv"
OUTPUT_FILE = "speech_features_from_combined.csv"

analyzer = SentimentIntensityAnalyzer()

LABEL_TO_PARTY = {
    0: "Democrat",
    1: "Republican"
}

In [3]:

KEYWORD_GROUPS = {
    "kw_democracy": {
        "democracy", "constitution", "rights", "liberty", "freedom",
        "justice", "equality", "vote", "voting", "citizen", "union"
    },
    "kw_economy": {
        "economy", "economic", "jobs", "job", "workers", "labor",
        "trade", "business", "inflation", "tariff", "budget",
        "debt", "deficit", "tax", "taxes", "wages"
    },
    "kw_welfare_health": {
        "health", "healthcare", "medical", "care", "insurance",
        "welfare", "poverty", "housing", "families", "children"
    },
    "kw_education": {
        "education", "school", "schools", "student", "students",
        "teacher", "teachers", "college", "university", "research"
    },
    "kw_environment": {
        "environment", "climate", "energy", "pollution", "water",
        "land", "conservation", "sustainable"
    },
    "kw_security_defense": {
        "security", "defense", "military", "army", "navy", "war",
        "peace", "terror", "threat", "violence", "crime", "border"
    },
    "kw_foreign_policy": {
        "foreign", "international", "world", "allies", "ally",
        "diplomacy", "treaty", "global", "nations", "abroad"
    },
    "kw_immigration": {
        "immigration", "immigrant", "immigrants", "migrant",
        "migrants", "citizenship", "refugee", "refugees", "border"
    }
}

WORD_RE = re.compile(r"\b[a-zA-Z']+\b")
SENTENCE_SPLIT_RE = re.compile(r"[.!?]+")

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\xa0", " ")
    text = text.replace("&nbsp;", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [5]:
def tokenize(text):
    return WORD_RE.findall(text.lower())


In [6]:
def avg_sentence_len(text):
    parts = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    if not parts:
        return 0.0
    lengths = [len(WORD_RE.findall(s.lower())) for s in parts]
    return sum(lengths) / len(lengths)


In [7]:
def keyword_features(tokens):
    total_words = len(tokens)
    counts = {name: 0 for name in KEYWORD_GROUPS}

    for tok in tokens:
        for group_name, vocab in KEYWORD_GROUPS.items():
            if tok in vocab:
                counts[group_name] += 1

    out = {}
    for group_name, count in counts.items():
        out[f"{group_name}_count"] = count
        out[f"{group_name}_freq"] = count / total_words if total_words > 0 else 0.0
    return out

In [8]:
speech = pd.read_csv(INPUT_FILE)

rows = []

for i, row_in in speech.iterrows():
    text = clean_text(row_in["Text"])
    label = row_in["Label"]

    if label not in LABEL_TO_PARTY:
        continue

    party = LABEL_TO_PARTY[label]
    tokens = tokenize(text)
    sentiment = analyzer.polarity_scores(text)

    row = {
        "speech_id": i + 1,
        "party": party,
        "sentiment_score": sentiment["compound"],
        "negativity_score": sentiment["neg"],
        "speech_length": len(tokens),
        "avg_sentence_length": avg_sentence_len(text)
    }

    row.update(keyword_features(tokens))
    rows.append(row)


FileNotFoundError: [Errno 2] No such file or directory: 'ultimate_combined_speeches.csv'

In [ ]:
df_features = pd.DataFrame(rows)
df_features.to_csv(OUTPUT_FILE, index=False)

print(df_features.head())
print(df_features.shape)
print(df_features["party"].value_counts())